In [17]:
import pandas as pd
import numpy as np
import xgboost
import os
from dotenv import load_dotenv
import json

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [3]:
load_dotenv()
path = os.getenv("DATA_PATH")
transaction_path = os.path.join(path, r'raw/train_transaction.csv/train_transaction.csv')
identity_path = os.path.join(path, r'raw/train_identity.csv/train_identity.csv')
train_transaction = pd.read_csv(transaction_path)
train_identity = pd.read_csv(identity_path)

In [4]:
df = train_transaction.merge(train_identity, on="TransactionID", how = "left")

In [5]:
df.shape
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,19.0,NaN,NaN,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0,0.0,1.0,1.0,14.0,NaN,13.0,NaN,NaN,NaN,NaN,NaN,NaN,13.0,13.0,NaN,NaN,NaN,0.0,T,T,T,M2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,M0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0,87.0,287.0,NaN,outlook.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.0,315.0,NaN,NaN,NaN,315.0,T,T,T,M0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,87.0,NaN,NaN,yahoo.com,NaN,2.0,5.0,0.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0,1.0,0.0,25.0,1.0,112.0,112.0,0.0,94.0,0.0,NaN,NaN,NaN,NaN,84.0,NaN,NaN,NaN,NaN,111.0,NaN,NaN,NaN,M0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,NaN,NaN,gmail.com,NaN,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.0,NotFound,NaN,-480.0,New,NotFound,166.0,NaN,542.0,144.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,New,NotFound,Android 7.0,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [6]:
df = df.sort_values("TransactionDT").reset_index(drop=True)

In [7]:
df["TransactionDT"].is_monotonic_increasing

True

In [8]:
length = len(df)
train_end = int(length * 0.70)
val_end = int(length * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

In [9]:
print("Train", train_df.shape)
print("Validation", val_df.shape)
print("Test", test_df.shape)

Train (413378, 434)
Validation (88581, 434)
Test (88581, 434)


In [10]:
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(name, df['isFraud'].mean())

train 0.03516878014795175
validation 0.03434145019812375
test 0.03480430340592226


In [11]:
base_features = ['TransactionDT', "TransactionAmt", "ProductCD", "P_emaildomain", "R_emaildomain"]
extended_features = base_features + ["card2", "card4", "card6", "addr1", "addr2", "dist1", "dist2"]
symbols = ['V', 'D', 'M', 'C', 'id', "card"]
def custom_features(df):

    df_custom = df[extended_features].copy()

    df_custom['missing_count'] = df.isna().sum(axis=1)

    for i in symbols:
        df_custom[f'missing_{i}_count'] = df.filter(regex = f"^{i}").isna().sum(axis=1)

    df_custom["transaction_day"] = df["TransactionDT"] // 86400

    df_custom["transaction_hour"] = (df["TransactionDT"] % 86400) // 3600

    return df_custom

In [12]:
names = ['train', "val", 'test']
features = ['base', 'extended']
X_datasets = {}
y_datasets = {}

df_map = {
    'train': train_df,
    'val': val_df,
    'test': test_df
}

feature_map = {
    'base': base_features,
    'extended': extended_features
}

X_datasets.clear()
y_datasets.clear()

for name in names:
    df = df_map[name]

    y_key = f"y_{name}" 
    y_datasets[y_key] = df['isFraud']
    
    for feature in features:
        feat_list = feature_map[feature]
        
        X_key = f"X_{name}_{feature}"

        X_datasets[X_key] = df[feat_list]

        if feature == "extended":
            X_datasets[f'X_{name}_custom'] = custom_features(df)

custom_features_names = X_datasets['X_train_custom'].columns

In [13]:
custom_features_names

Index(['TransactionDT', 'TransactionAmt', 'ProductCD', 'P_emaildomain',
       'R_emaildomain', 'card2', 'card4', 'card6', 'addr1', 'addr2', 'dist1',
       'dist2', 'missing_count', 'missing_V_count', 'missing_D_count',
       'missing_M_count', 'missing_C_count', 'missing_id_count',
       'missing_card_count', 'transaction_day', 'transaction_hour'],
      dtype='object')

In [14]:
feature_sets = {
    "baseline": base_features,
    "transaction": extended_features,
    "engineered": custom_features_names
}

In [15]:
for name, feature in feature_sets.items():
    print(name, len(feature))

baseline 5
transaction 12
engineered 21


In [16]:
split_info = {
    "train_end": train_end,
    "validation_end": val_end,
    "train_size": len(train_df),
    "validation_size": len(val_df),
    "test_size": len(test_df),
    "train_max_dt": int(train_df["TransactionDT"].max()),
    "validation_max_dt": int(val_df["TransactionDT"].max()),
    "test_max_dt": int(test_df["TransactionDT"].max())
}

In [20]:
processed_path = os.path.join(path, "processed/split_info.json")

with open (processed_path, "w") as f:
    json.dump(split_info, f, indent=4)